ollama list
ollama run qwen3.5:9b
ollama pull nomic-embed-text
pip install llama-index-core llama-index-readers-file llama-index-llms-ollama llama-index-embeddings-ollama llama-index-vector-stores-chroma chromadb

In [8]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from llama_index.core.tools import FunctionTool
from llama_index.core.tools import QueryEngineTool

from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    ReActAgent,
)
 
# -------------------------------------------------
# 1. Load documents
# -------------------------------------------------

documents = SimpleDirectoryReader(
    input_dir="./data"
).load_data()

print(f"Loaded {len(documents)} documents")

Loaded 1 documents


# 2. Connect to local Ollama models

In [9]:
llm = Ollama(
    model="qwen3.5:9b",
    request_timeout=300.0,
    context_window=8192,
)

embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
)

# 3. Create persistent Chroma database


In [10]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

chroma_collection = (
    chroma_client.get_or_create_collection(
        name="industrial_documents"
    )
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

# 4. Split, embed, and store documents


In [13]:
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(
            chunk_size=512,
            chunk_overlap=50,
        ),
        embed_model,
    ],
    vector_store=vector_store,
)

nodes = pipeline.run(
    documents=documents,
    show_progress=True,
)

print(f"Created and stored {len(nodes)} nodes")

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

2026-08-06 18:46:02,892 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-06 18:46:03,045 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Created and stored 11 nodes


# 5. Create the searchable index


In [14]:
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)

# 6. Create the query engine


In [15]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=4,
    response_mode="compact",
)


# 7. Ask a question

In [16]:
response = query_engine.query(
    "Explain . What is weight."
)

print(response)

2026-08-06 18:47:45,076 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-06 18:49:40,876 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Weight is defined as gravitational force acting upon an object. It differs from mass because it represents a downward force rather than just quantity of matter; for instance, while kilogram measures mass, weight must be expressed in newtons when performing SI calculations regarding force. To calculate this value, you multiply the mass by gravity ($W=mg$).

In practical applications like load-cell systems, the device responds to approximately 245 N of downward force generated by a 25 kg calibration mass due to gravitational acceleration (9.81), rather than reading merely as "25". This distinction is critical because equipment calibrated for one specific density or liquid will not read correctly on denser or lighter fluids unless factors like gravity and mass are properly distinguished in the equation. Consequently, weight involves a direct relationship with displacement where force relates to acceleration over time ($F = m d^2x/dt^2$), distinguishing it from static mass readings alone.


In [5]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    ReActAgent,
    
)
from llama_index.core.workflow import Context 

In [18]:
def multiply(a: int, b: int) -> int:
    """Multiplies two integers and returns the resulting integer"""
    return a * b

def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract, multiply],
    llm=llm,
)

# query_agent = ReActAgent(
#     name="info_lookup",
#     description="Looks up information about XYZ",
#     system_prompt="Use your tool to query a RAG system to answer information about XYZ",
#     tools=[query_engine_tool],
#     llm=llm
# )

In [24]:
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="name",
    description="a specific description",
    return_direct=False,
)
agent = AgentWorkflow.from_tools_or_functions(
    [FunctionTool.from_defaults(multiply),query_engine_tool],
    llm=llm
)
ctx = Context(agent)

In [25]:
response = await agent.run(
    "What is multiply 4 and 6?"
)

print(response)
    

2026-08-06 18:58:34,568 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-08-06 18:58:43,718 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Multiply 4 and 6 equals **24**.


In [26]:
response = await agent.run(
    "My name is Ammar.",
    ctx=ctx
)

2026-08-06 18:59:16,836 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


In [27]:

response = await agent.run(
    "what is my name",
    ctx=ctx
)
print(response)

2026-08-06 19:00:26,403 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Your name is Ammar! Is there anything else you'd like to ask or talk about?


In [28]:

response = await agent.run(
    "Are you good robot",
    ctx=ctx
)
print(response)

2026-08-06 19:03:14,380 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Thank you! As an AI virtual assistant, I'm designed to provide support and help answer your questions. How can I assist you today?
